# v0.3 S2-HP — readers on HotpotQA (E1 + E2): Qwen and Llama in parallel
**How to run:** Settings → Accelerator **GPU T4 x2**, Internet **On**. Then **Save Version → Save & Run All (Commit)**. It runs unattended (about 4–5 h). If a step fails, the notebook stops with `FAILED` — send me the log. When it finishes, open the version's **Output** tab, download `v03_s2hp_results.zip` and send it to me.

**Input:** Add Input → **Your Work → Notebooks** → the S1-HP notebook (its output `v03_s1hp_results.zip`). Secrets: **HF_TOKEN** on.

In [ ]:
# Cell 0 — helper: run a shell command, stream its output, STOP the notebook if it fails
import os, subprocess
def sh(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")

In [ ]:
# Cell 0b — run two commands at once (one per GPU); STOP if either fails
import threading
def sh_par(*cmds):
    errs = []
    def run(c):
        try:
            sh(c)
        except Exception as e:
            errs.append(str(e))
    ts = [threading.Thread(target=run, args=(c,)) for c in cmds]
    [t.start() for t in ts]; [t.join() for t in ts]
    if errs:
        raise RuntimeError("; ".join(errs))

In [ ]:
# Cell 0c — the code of commit b561147 as base64 tar.gz (do not edit; Cell 1 checks it)
CODE = "..."

In [ ]:
# Cell 1 — hardware, code frozen at b561147 (embedded in Cell 0c), pinned environment, check, tests
sh("nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv || true")
sh("df -h / /tmp /kaggle/working && free -g && nproc")
os.chdir("/kaggle/working")
try:                                   # Hugging Face token from Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN)
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("HF_TOKEN loaded")
except Exception as e:
    print("HF_TOKEN not available (public downloads still work):", type(e).__name__)
# the GitHub repo is private: the code of commit b561147b1c7fb007faf7244da62e7d52ed17fe8d is embedded in Cell 0c (no dataset, no clone);
# STOP unless the extracted files reproduce that commit's git tree exactly (paper_inputs, paper_outputs left out)
import base64, io, shutil, tarfile
shutil.rmtree("rag", ignore_errors=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE))).extractall("rag")
os.chdir("/kaggle/working/rag")
sh("git init -q && git add -A -f")
sh("test \"$(git write-tree)\" = 6ccd006ec0ae1b8e16b2ad6a94478df45bf18a85 || (echo WRONG CODE; exit 1)")
sh("git -c user.name=kaggle -c user.email=kaggle@local commit -qm 'code b561147b1c7fb007faf7244da62e7d52ed17fe8d' && echo CODE OK b561147")
sh("pip install -q -r requirements-v03-retrieval.txt")
sh("pip install -q -r requirements-v03-generation.txt")

sh("python -m pytest -q -p no:cacheprovider 2>&1 | tail -3")

In [ ]:
# Cell 2 — restore the S1-HP output FIRST (nothing writes results/ before this), then environment record,
# then the corpus text + qrels at the SAME pinned revisions as S1 (needed for passage text and the retrieval summary)
sh("python -m src.restore --name v03_s1hp_results")
sh("python -m src.check_env --lock requirements-generation.lock --report-only results/v03_hp/raw/check_env_v03_S2hp.json")
sh("python -m src.env_report --session v03_S2hp --out results/v03_hp/raw/env_v03_S2hp.json")
sh("python -m src.data.download --config configs/v03_hp_qwen.yaml --only hp_corpus,hp_qrels")

In [ ]:
# Cell 3 — E1 retrieval summary (descriptive): Hit@5, Both@5, nDCG@10 per retriever, all 7,405 queries + the 2,000 generation queries
sh("python -m src.analysis.retrieval_summary --config configs/v03_hp_qwen.yaml --out summary/v03_hp_retrieval_summary.json")

In [ ]:
# Cell 4 — E1 + E2 readers in parallel: GPU 0 = Qwen2.5-7B-AWQ, GPU 1 = Llama-3.1-8B-AWQ; 9 conditions x 2,000 queries each (~3-4 h)
GEN_ERR = ""
try:
    sh_par("CUDA_VISIBLE_DEVICES=0 python -m src.generation.run --config configs/v03_hp_qwen.yaml --conditions R0,R1,R2,R3,R4,R5,R6,CB,GOLD --corpus main --n 2000 --k 5 --auto-eager-retry",
           "CUDA_VISIBLE_DEVICES=1 python -m src.generation.run --config configs/v03_hp_llama.yaml --conditions R0,R1,R2,R3,R4,R5,R6,CB,GOLD --corpus main --n 2000 --k 5 --auto-eager-retry")
except Exception as e:
    GEN_ERR = str(e); print("READER FAILURE (recorded; finished conditions are still packaged):", GEN_ERR)

In [ ]:
# Cell 5 — score answers (EM, F1, answer containment) for both readers
sh("python -m src.evaluation.answer --config configs/v03_hp_qwen.yaml --prefix v03_hp_qwen")
sh("python -m src.evaluation.answer --config configs/v03_hp_llama.yaml --prefix v03_hp_llama")

In [ ]:
# Last cell — package: everything of v0.3-HP EXCEPT the large run files (they stay in the S1 output) -> v03_s2hp_results.zip
# small (about 10-30 MB): send this file to Claude; the next sessions attach S1 + S2 outputs
import zipfile
from pathlib import Path
n = 0
with zipfile.ZipFile("/kaggle/working/v03_s2hp_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for k in ["results/v03_hp", "results/raw/restore.jsonl", "summary", "data/processed", "configs/resolved_revisions.json", "logs"]:
        for p in ([Path(k)] if Path(k).is_file() else sorted(Path(k).rglob("*"))):
            if p.is_file() and "/runs/" not in p.as_posix() and (not k.startswith("data/") or p.name.startswith("hp_join") or p.name.startswith("hp_query")):
                z.write(p); n += 1
print(n, "files"); sh("ls -la /kaggle/working/v03_s2hp_results.zip")
if GEN_ERR:
    raise RuntimeError("FAILED (packaged anyway): " + GEN_ERR)